In [ ]:
# Jason's implementation of directional algo
from algorithim import Params, Algorithm, Capon
from scipy.signal import find_peaks

import os
os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/x86_64-linux-gnu/libLLVM.so:'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

import os # Configure which GPU

if os.getenv("CUDA_VISIBLE_DEVICES") is None:
    gpu_num = 0 # Use "" to use the CPU
    os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu_num}"
    print(f"Using GPU {gpu_num}")
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

# Import Sionna
try:
    import sionna
except ImportError as e:
    # Install Sionna if package is not already installed
    import os
    os.system("pip install sionna")
    import sionna

# Configure the notebook to use only a single GPU and allocate only as much memory as needed
# For more details, see https://www.tensorflow.org/guide/gpu
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))
gpus = tf.config.list_physical_devices('GPU')
print(f"Found {len(gpus)} GPUs")
if gpus:
    print(f"Found GPU: {gpus[0].name}")
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)

# Avoid warnings from TensorFlow
tf.get_logger().setLevel('ERROR')

# Colab does currently not support the latest version of ipython.
# Thus, the preview does not work in Colab. However, whenever possible we
# strongly recommend to use the scene preview mode.
try: # detect if the notebook runs in Colab
    import google.colab
    no_preview = True # deactivate preview
except:
    if os.getenv("SIONNA_NO_PREVIEW"):
        no_preview = True
    else:
        no_preview = False

resolution = [480,320] # increase for higher quality of renderings

# Define magic cell command to skip a cell if needed
from IPython.core.magic import register_cell_magic
from IPython import get_ipython

@register_cell_magic
def skip_if(line, cell):
    if eval(line):
        return
    get_ipython().run_cell(cell)

# Set random seed for reproducibility
sionna.config.seed = 42

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

# Import Sionna RT components
from sionna.rt import load_scene, Transmitter, Receiver, PlanarArray, Camera, AntennaArray, Antenna

# For link-level simulations
from sionna.channel import cir_to_ofdm_channel, subcarrier_frequencies, OFDMChannel, ApplyOFDMChannel, CIRDataset
from sionna.nr import PUSCHConfig, PUSCHTransmitter, PUSCHReceiver
from sionna.utils import compute_ber, ebnodb2no, PlotBER
from sionna.ofdm import KBestDetector, LinearDetector
from sionna.mimo import StreamManagement
from sionna.ofdm import ResourceGrid

def calculate_heading_rad(start, end):
    # Calculate the difference vector
    diff = np.array(end) - np.array(start)
    
    # Calculate the heading angle in radians
    heading = np.arctan2(diff[1], diff[0])
    
    return heading

def get_rss_from_csi(csi):
  # Compute CSI power (power per subcarrier)
  csi_power = tf.abs(csi)**2

  # Compute RSS per receiver-transmitter pair (sum over subcarriers)
  rss_per_rx_tx = tf.reduce_sum(csi_power, axis=-1)  # Sum over subcarriers

  # Convert RSS from linear scale (e.g., Watts) to dBm
  rss_per_rx_tx_dBm = 10 * tf.math.log(rss_per_rx_tx / 1e-3) / tf.math.log(10.0)

  return rss_per_rx_tx_dBm

def dbm_to_watts(dbm):
    return 10. ** ((dbm-30)/10)

def watts_to_dbm(watts):
    epsilon = 1e-10  # Small constant to avoid log(0)
    return 10 * np.log10(watts + epsilon) + 30

# # Multiple Antennas like ASUS Router
def get_antenna_positions(spacing):
    return np.array([[0.0, spacing / 2 + spacing , 0.0],
                     [0.0, spacing / 2, 0.0],
                     [0.0, -spacing / 2, 0.0],
                     [0.0, -spacing / 2 - spacing, 0.0]])

def flip_trajectory(data):
    # Vector from origin to the first and last points of each trajectory
    first_points = data[:, 0, :]  # Shape (1000, 3)
    last_points = data[:, -1, :]  # Shape (1000, 3)

    # Compute the dot product of the first and last points with respect to the origin
    dot_products = np.sum(first_points * last_points, axis=1)

    # Identify trajectories heading away from the origin (dot product > 0)
    away_from_origin = dot_products < 0

    # Flip the trajectories that are not heading away from the origin
    data[~away_from_origin] = data[~away_from_origin, ::-1, :]

    return data

def path_loss_db(distance_m, frequency_hz):
    """
    Calculate free space path loss in dB.

    Parameters:
    - distance_m: distance between antennas in meters
    - frequency_hz: signal frequency in Hz

    Returns:
    - path loss in dB
    """
    c = 3e8  # speed of light in m/s
    loss_db = 20 * np.log10(distance_m) + 20 * np.log10(frequency_hz) - 147.55
    return loss_db

In [ ]:
# List for dataset
angles = []
angle_profile_values = []
rssi = []
rssi_normalized = []

total_traj = 100

# SCENE_NAME = "meshes_512/small_scene0"
SCENE_NAME = "sharp_terrain/scenes/lunar_terr_0_0_scene"
IMAGE_FOLDER = f"images/{SCENE_NAME}"

# Load integrated scene
scene = load_scene(f"models/{SCENE_NAME}.xml") # Try also sionna.rt.scene.etoile

# Configure antenna array for all transmitters
scene.tx_array = PlanarArray(num_rows=1,
                             num_cols=1,
                             vertical_spacing=0.5,
                             horizontal_spacing=0.5,
                             pattern="dipole",
                             polarization="V")

antenna_positions = get_antenna_positions(0.03)

antenna_array_angle = -175 # Default

RelativeAntennas = AntennaArray(antenna=Antenna("dipole", "V"), 
                            positions=tf.Variable(antenna_positions.tolist()))

scene.rx_array = RelativeAntennas

# Create transmitter
tx_position = [0., 15., 7.87 + 0.5] # 26.5 + 2.5]  # Adjusted to be closer to the ground
tx = Transmitter(name="tx",
                 position=tx_position,
                 orientation=[np.radians(0),0,0])

scene.add(tx)

# trajectories = np.load('small_mesh0_2.npy')
# trajectories = np.load("small_mesh0_2-short.npy")
trajectories = np.load('terr_0_0_2.npy')[:,:,:]
print(f"Loaded trajectories with shape: {trajectories.shape}")

trajectories = trajectories[:,:total_traj,:]
trajectory_index = 2 #103 # The one close to tx 2
d = np.linalg.norm(np.array(tx_position) - trajectories[trajectory_index, 0, :]) # meters
points = trajectories[trajectory_index, :, :]  # shape: [T, 3]
diffs = np.diff(points, axis=0)                # shape: [T-1, 3]
segment_lengths = np.linalg.norm(diffs, axis=1)  # shape: [T-1]
total_distance = np.sum(segment_lengths)
print(f"Total distance: {total_distance:.2f} m")
print(f"Distance from transmitter to trajectory {trajectory_index}: {d:.2f} m")
f =  5.745e9  # 2.4 GHz
pl_db = path_loss_db(d, f)
print(f"Path loss at {d}m and {f/1e9}GHz: {pl_db:.2f} dB")
print(f"Transmit power (dBm): {tx.power_dbm} dBm")
received_power_fspl = tx.power_dbm - pl_db
print(f"Received power at trajectory {trajectory_index} (dBm): {received_power_fspl:.2f} dBm")
num_steps = trajectories.shape[1]
trajectory_steps =  np.arange(num_steps - 1)# [10, 12, 14, 16, 18]

# Calculate headings for the trajectory
calculated_headings = []
for i in trajectory_steps: #range(0, len(trajectories[0]) - 1):
    heading = calculate_heading_rad(trajectories[trajectory_index,i,:2], trajectories[trajectory_index,i + 1,:2])
    calculated_headings.append(heading)

rxs = []
for i, position in enumerate(trajectories[trajectory_index, np.arange(num_steps),:]):
    if i == (trajectories[0].shape[0] - 1):
        heading = calculated_headings[-1]
    else:
        heading = calculated_headings[i]
    rxs.append(Receiver(name=f"rx{i}",
              position=position,
              orientation=[heading,0,0]))    
    scene.add(rxs[i])

antenna_array_angle = np.radians(180) # heading

# Select an example object from the scene
# so = scene.get('itu_concrete')
obj = scene.get("ground")
# obj.radio_material.relative_permittivity = 1
obj.radio_material.scattering_coefficient = 1/np.sqrt(3)
from sionna.rt import LambertianPattern, DirectivePattern, BackscatteringPattern
obj.radio_material.scattering_pattern = LambertianPattern() #DirectivePattern(alpha_r=10)

# print(obj.radio_material.name)
# print(f"\nRadioMaterial:", obj.radio_material.conductivity.numpy())
# print("Relative permittivity:", obj.radio_material.relative_permittivity.numpy())
# print("Complex relative permittivity:", obj.radio_material.complex_relative_permittivity.numpy())
# print("Relative permeability:", obj.radio_material.relative_permeability.numpy())
# print("Scattering coefficient:", obj.radio_material.scattering_coefficient.numpy())
# print("Scattering Pattern:", obj.radio_material.scattering_pattern)
# print("XPD coefficient:", obj.radio_material.xpd_coefficient.numpy())

scene.frequency = 5.745e9 #5.745 # in Hz; implicitly updates RadioMaterials

scene.synthetic_array = False # If set to False, ray tracing will be done per antenna element (slower for large arrays)
"When the property scene.synthetic_array is set to False, antenna arrays are explicitly modeled by finding paths between any pair of transmitting and receiving antennas in the scene. Otherwise, arrays are represented by a single antenna located in the center of the array. Phase shifts related to the relative antenna positions will then be applied based on a plane-wave assumption when the channel impulse responses are computed"

paths = scene.compute_paths(max_depth=2,
                            num_samples=1e5,  # Number of rays shot into directions defined
                                              # by a Fibonacci sphere , too few rays can
                                              # lead to missing paths
                            method='fibonacci',  # Method to sample directions
                            
                            los=True,  # Include Line-of-Sight paths
                            reflection=True,  # Include reflection paths
                            diffraction=True,  # Include diffraction paths
                            scattering=True,  # Include scattering paths
                            edge_diffraction=True,  # Include edge diffraction paths
                            ris=False,  # Reflecting Intelligent Surfaces (RIS) are not considered in this example
                            scat_random_phases=True,  # Randomize phases of scattering paths
                            )

paths.normalize_delays = False

# Determine subcarrier frequencies
rg = ResourceGrid(num_ofdm_symbols=1,
                fft_size=52,
                dc_null = True,
                cyclic_prefix_length=20,
                #   pilot_pattern = "kronecker",
                #   pilot_ofdm_symbol_indices = [2, 8],
                subcarrier_spacing=5e6) #30e3)

frequencies = subcarrier_frequencies(rg.fft_size, rg.subcarrier_spacing)

# get rss
a, tau = paths.cir()

csi = cir_to_ofdm_channel(frequencies, a, tau, normalize=False)  # Non-normalized includes path-loss

csi_reshaped = []
for i in range(len(rxs)):
    csi_reshaped.append(np.array(csi[:,i,:,:,:,:,:]).reshape(4, 52))

#Output: h_f ([batch size, num_rx, num_rx_ant, num_tx, num_tx_ant, num_time_steps, fft_size], tf.complex) – Channel frequency responses at frequencies
block_rss = get_rss_from_csi(csi).numpy()[0, :, 0, 0, 0, 0]
noise_floor = -250
block_rss[np.isneginf(block_rss)] = noise_floor
block_rss[block_rss < noise_floor] = noise_floor # Set a floor for RSS values to avoid extreme values
print(f"block_rss {block_rss}")
# Define range
min_rss = noise_floor
max_rss = 0
# Normalize values to 0-1
normalized = (block_rss - min_rss) / (max_rss - min_rss)
print(f"rssi normalized {normalized}")

# lets make a graph with CSI as time progresses (assuming time progresses when the Rx has moved/changed spatially) 
# we have csi_reshaped with dims (20, 4, 52)
# 20 refers to the time/space/each Rx, 4 is the num of Rx antennas, 52 is the different subcarrier freqs
# and the data stored itself is the complex number at Rx that .angle = phaseshift, .mag = amplitude -> dB
csi_reshaped = np.array(csi_reshaped)

# need this for amplitude to dB
def dB(x):
    return 10 * np.log10(np.abs(x) ** 2)
    # return 10 * np.log10(np.maximum(np.abs(x) ** 2, 1e-8))

# lets say we only care about Rx antenna #0
C = dB(csi_reshaped[:, 0, :])

# csi: shape (N_subcarriers, N_rx, N_tx), complex-valued
# |H|^2 gives power per subcarrier
csi_magnitude_squared = np.abs(csi_reshaped[0,:,:])**2
power_per_subcarrier = csi_magnitude_squared.mean(axis=(0))  # average over Tx/Rx
avg_csi_power_db = 10 * np.log10(np.mean(power_per_subcarrier))
print(f"Average CSI Power (dB): {avg_csi_power_db:.2f} dB") 
print(f"Power {tx.power_dbm + avg_csi_power_db} dBm")  # Compare with received power


plt.pcolormesh(np.arange(csi_reshaped.shape[0]), np.arange(52), C.T)
plt.xticks(range(20))
plt.title(f"csi magnitudes as rx moves / time changes")
plt.ylabel("subcarrier index")
plt.xlabel("time / rx number")
plt.colorbar(orientation='vertical', fraction=0.02, pad=0.04, label="CSI Magnitude (dB)")

scene.preview(paths, show_devices=True, show_paths=True, show_orientations=True)

In [ ]:
# print(f"Trajectory length {np.linalg.norm(trajectories[trajectory_index, :, :])}")
# Select the trajectory (shape [T, 3])
points = trajectories[trajectory_index, :, :]  # shape: [T, 3]

# Compute pairwise differences between consecutive points
diffs = np.diff(points, axis=0)  # shape: [T-1, 3]

# Compute Euclidean distances for each segment
segment_lengths = np.linalg.norm(diffs, axis=1)  # shape: [T-1]

# Total distance
total_distance = np.sum(segment_lengths)
print(f"Total distance: {total_distance:.2f} m")

In [ ]:
import trimesh
# mesh = trimesh.load_mesh("models/meshes_512/small_mesh0.ply")
mesh = trimesh.load_mesh("models/sharp_terrain/scenes/lunar_terr_0_0_mesh.ply")
print(normalized.shape)
radius = .50  # small radius for visibility
markers = []
for rssi, pose in zip(normalized, trajectories[trajectory_index,:,:]):
    marker = trimesh.creation.icosphere(radius=radius, color=[rssi, 0, 1 - rssi, 1])  # Normalize rssi to 0-1
    marker.apply_translation(pose)
    markers.append(marker)

scene = trimesh.Scene()
scene.add_geometry(markers)
scene.add_geometry(mesh)
scene.show()

In [ ]:
# Assumes CSI is real or complex magnitude
csi_mag = np.abs(csi_reshaped[:, 0, :])  # or use np.sqrt(real**2 + imag**2)
csi_log = np.log10(csi_mag + 1e-12)  # avoid log(0)
# Normalize to zero mean, unit variance
mean = np.mean(csi_log)
std = np.std(csi_log)
data_csi = (csi_log - mean) / std

plt.pcolormesh(np.arange(csi_reshaped.shape[0]), np.arange(52), data_csi.T)
plt.xticks(range(20))
plt.title(f"csi magnitudes as rx moves / time changes")
plt.ylabel("subcarrier index")
plt.xlabel("time / rx number")
plt.colorbar(orientation='vertical', fraction=0.02, pad=0.04, label="CSI Magnitude (dB)")